# CS 3110/5110: Data Privacy
## Homework 8

In [1]:
# Load the data and libraries
import pandas as pd
import numpy as np
import random
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def laplace_mech_vec(vec, sensitivity, epsilon):
    return [v + np.random.laplace(loc=0, scale=sensitivity / epsilon) for v in vec]

def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)

def gaussian_mech_vec(vec, sensitivity, epsilon, delta):
    return [v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)
            for v in vec]

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

In [2]:
# preserves epsilon-differential privacy
def above_threshold(queries, df, T, epsilon):
    T_hat = T + np.random.laplace(loc=0, scale = 2/epsilon)
    
    for idx, q in enumerate(queries):
        nu_i = np.random.laplace(loc=0, scale = 4/epsilon)
        if q(df) + nu_i >= T_hat:
            return idx
    return -1 # the index of the last element

## Question 1 (20 points)

Implement a function `above_10000` that releases the **value** of the first query in a sequence of queries whose value is above 10000. Your function should have a **total** privacy cost equal to the privacy parameter $\epsilon$ passed in when it is called.

In [3]:
def make_query(status):
    def q(df):
        return len(df[df['Marital Status'] == status])
    return q

def above_10000(queries, epsilon):
    T = 1000
    idx = above_threshold(queries, adult, T=T, epsilon=epsilon/2)
    qeuery_value = queries[idx](adult)
    noisy_value = laplace_mech(qeuery_value, sensitivity=1, epsilon=1/2)
    return noisy_value
    

queries = [make_query(status) for status in adult['Marital Status'].unique()]
print(f"above_10000 #1: {above_10000(queries, 100)}")
print(f"above_10000 #2: {above_10000(queries, 1)}")
print(f"above_10000 #3: {above_10000(queries, .01)}")

above_10000 #1: 10679.445655588745
above_10000 #2: 10682.484553395343
above_10000 #3: 10680.878710279485


In [4]:
# TEST CASE

results = [above_10000(queries, 1.0) for _ in range(100)]
print(np.mean(results))
assert np.mean(results) > 9900
assert np.mean(results) < 11000

10682.694165907717


## Question 2 (10 points)
In 2-3 sentences, argue informally (via the definition of the sparse vector technique, post-processing, and sequential composition), that your implementation of `above_10000` has a total privacy cost of $\epsilon$.

YOUR ANSWER HERE

## Question 3 (20 points)

Implement a function `bounded_all_above_10000` that releases the **value** of **$c$ queries** in a sequence of queries whose value is above 10000 (where $c$ is an analyst-provided parameter limiting the number of returned results). Your function should have a **total privacy cost** bounded by its parameter $\epsilon$.

In [5]:
def bounded_all_above_10000(queries, c, epsilon):
    T = 1000
    queries_temp_in = queries
    noisy_vals = []
    epsilon_c = epsilon/c
    for _ in range(c):
        if not queries_temp_in:
            break
        
        idx = above_threshold(queries, adult, T=T, epsilon=epsilon_c/2)

        if idx == -1:
            break

        true_val = queries_temp_in[idx](adult)
        noisy_val = laplace_mech(true_val, sensitivity=1, epsilon=epsilon_c/2)
        noisy_vals.append(noisy_val)

        queries_temp_in = queries_temp_in[idx + 1:]

    
    return noisy_vals

queries = [make_query(status) for status in adult['Marital Status'].unique()]
print(f"bounded_all_above_10000 #1: {bounded_all_above_10000(queries, 3, 100)}")
print(f"bounded_all_above_10000 #2: {bounded_all_above_10000(queries, 3, 1)}")
print(f"bounded_all_above_10000 #3: {bounded_all_above_10000(queries, 3, .01)}")

bounded_all_above_10000 #1: [10683.024997368126, 14975.966313608045, 4442.848623345327]
bounded_all_above_10000 #2: [10689.077643200982, 14975.9449292078, 4435.6754484146]
bounded_all_above_10000 #3: [11134.931905478781, 15422.7252767875, 5539.644771494262]


In [6]:
# TEST CASE

results = [bounded_all_above_10000(queries, 2, 1.0) for _ in range(100)]
results_1 = [r[0] for r in results]
results_2 = [r[1] for r in results]

assert np.mean(results_1) > 9900
assert np.mean(results_1) < 11000
assert np.mean(results_2) > 14000
assert np.mean(results_2) < 15500

## Question 4 (10 points)

In 2-3 sentences, argue informally that your implementation of `bounded_all_above_10000` has privacy cost bounded by $\epsilon$.

YOUR ANSWER HERE

## Question 5 (30 points)

Implement a function `mean_age` that computes the mean age of participants in the `adult_data` dataset. Your function should have a **total** privacy cost of $\epsilon$. It should work as follows:

1. Compute an upper clipping parameter based on the data using a differentially private algorithm
2. Clip the data using the clipping parameter
3. Use `laplace_mech` to release a differentially private mean of the clipped data

*Hint*: Use the sparse vector technique (`above_threshold`) to compute the clipping parameter. Consider using a sequence of queries that looks like `df.clip(lower=0, upper=b).sum() - df.clip(lower=0, upper=b+1).sum()`.

*Hint*: Be careful of sensitivities and set the scale of the noise accordingly!

In [7]:
bs = list(range(0, 200, 10))
df = adult['Age']
b_lower = 0

def make_query(b):
    def q(df):
        return df["Age"].clip(lower=0, upper=b).sum() - df["Age"].clip(lower=0, upper=b+1).sum()
    return q

def mean_age(epsilon):
    queries = [make_query(b) for b in range(1, 1000)]
    b = above_threshold(queries, adult,T=0, epsilon=epsilon/2)
    true_mean = adult["Age"].clip(lower=0, upper=b).mean()
    # 1 person could change the sum by at most the clipping parameter (b)
    # so 1 person could change the mean by b / [# of adults]
    sens = b / len(adult) 
    noisy_mean = laplace_mech(true_mean, sensitivity= sens, epsilon=epsilon/2)
    return noisy_mean
    
for epsilon in [0.001, 0.01, 0.1, 0.5, 1, 10]:
    print(f"epsilon: {epsilon}, mean age: {mean_age(epsilon)}")

epsilon: 0.001, mean age: 23.3062267086722
epsilon: 0.01, mean age: 38.551752030590144
epsilon: 0.1, mean age: 38.57061330836317
epsilon: 0.5, mean age: 38.573729371121864
epsilon: 1, mean age: 38.570681549706464
epsilon: 10, mean age: 38.582644328013124


In [8]:
# TEST CASE
results = [mean_age(1.0) for _ in range(100)]
assert np.mean(results) > 38
assert np.mean(results) < 39

## Question 6 (10 points)

In 3-5 sentences, describe your approach to implementing `mean_age` and argue informally that your implementation has privacy cost bounded by $\epsilon$.

YOUR ANSWER HERE

## Question 7 (20 points)

Write an algorithm to compute the (approximate) maximum of a given column of the adult dataset. Your solution should satisfy `epsilon` differential privacy.

*Hint*: you might want to use AboveThreshold; you might need to test several options for building your queries and for the threshold

In [ ]:
def make_query(b, column):
    def q(df):
        # Number of people greater than b
        count = len(df[df[column] > b])

        # above_threshold finds the first query that is above the treshold, so to find the first query below the threshold multiply the count by -1
        # Values increase until they reach 0
        count *= -1
        return count
    return q
def max_of_col(column, epsilon):
    r = [2**(i/4) for i in range(22)] 

    queries = [make_query(b, column) for b in r]

    # Stops at the first query with 0 people
    idx = above_threshold(queries=queries, df=adult, T=0, epsilon=epsilon/2)
    true_val = r[idx]

    # Gap between 2 neighboring queries
    sens = r[-1] - r[-2]
    noisy_val = laplace_mech(true_val, epsilon=epsilon/2, sensitivity=sens)
    return noisy_val



max_of_col('Age', 1.0)

47.516969664206684

In [63]:
# TEST CASE
cols = ['Age', 'Capital Gain', 'fnlwgt']

for c in cols:
    true_val = adult[c].max()
    trials = [max_of_col(c, 10.0) for _ in range(20)]
    errors = [pct_error(true_val, t) for t in trials]
    print('Median error for column ' + c + ':', np.median(errors))
    assert np.median(errors) > 0
    assert np.median(errors) < 100

Median error for column Age: 57.36911093873718
Median error for column Capital Gain: 99.96212949897533
Median error for column fnlwgt: 99.99744824016592


## Question 8 (10 points)

In 3-5 sentences, describe your approach to implementing `max_of_col` and argue that your approach has total privacy cost bounded by $\epsilon$.

YOUR ANSWER HERE